# Emulating non-verbatim circuits with Qiskit

In a [previous notebook](https://github.com/amazon-braket/amazon-braket-examples/blob/main/examples/braket_features/Device_emulation/01_Local_Emulation_for_Verbatim_Circuits_on_Amazon_Braket.ipynb), we introduced how to emulate verbatim circuits locally based on device calibration data. Here, we show how to combine the [Qiskit-Braket provider](https://github.com/qiskit-community/qiskit-braket-provider) compilation pipeline with the local emulator to emulate non-verbatim circuits for a targeted QPU.

The workflow is:
1. Build a circuit using the **Braket SDK** with arbitrary gates.
2. Use `to_braket()` with the `braket_device` parameter to **transpile and convert** the circuit to a Braket verbatim circuit targeting the QPU's native gate set and connectivity.
3. Run the verbatim circuit on the Braket **local emulator** with a realistic noise model derived from device calibration data.

In [1]:
from braket.tracking import Tracker

t = Tracker().start()

## Choose a target QPU

Change `DEVICE_ARN` to retarget the entire notebook to a different QPU.

In [2]:
from braket.aws import AwsDevice
from braket.devices import Devices

# DEVICE_ARN = Devices.Rigetti.Cepheus1108Q
DEVICE_ARN = Devices.IQM.Garnet
# DEVICE_ARN = Devices.IonQ.Forte1
# DEVICE_ARN = Devices.IonQ.ForteEnterprise1

device = AwsDevice(DEVICE_ARN)
print(f"Device: {device.name}")
print(f"Native gates: {device.properties.paradigm.nativeGateSet}")
print(f"Qubits: {device.properties.paradigm.qubitCount}")

Device: Garnet
Native gates: ['cz', 'prx', 'cc_prx', 'measure_ff', 'barrier']
Qubits: 20


## Build a high-level circuit with the Braket SDK

We create a 4-qubit GHZ circuit using standard gates (`H`, `CNot`). These are **not** native to the target QPU and must be translated before verbatim execution.

In [3]:
from braket.circuits import Circuit

NUM_QUBITS = 4

braket_circuit = Circuit()
braket_circuit.h(0)
for i in range(NUM_QUBITS - 1):
    braket_circuit.cnot(i, i + 1)

print("Original Braket circuit:")
print(braket_circuit)

Original Braket circuit:
T  : │  0  │  1  │  2  │  3  │
      ┌───┐                   
q0 : ─┤ H ├───●───────────────
      └───┘   │               
            ┌─┴─┐             
q1 : ───────┤ X ├───●─────────
            └───┘   │         
                  ┌─┴─┐       
q2 : ─────────────┤ X ├───●───
                  └───┘   │   
                        ┌─┴─┐ 
q3 : ───────────────────┤ X ├─
                        └───┘ 
T  : │  0  │  1  │  2  │  3  │


## Compile to native gates with `to_braket()`

The `to_braket()` function from the [Qiskit-Braket provider](https://github.com/qiskit-community/qiskit-braket-provider) can accept a Braket circuit directly and, when given a `braket_device`, automatically:

1. Extracts the target device's native gate set and qubit connectivity.
2. Compiles the circuit to native gates with proper qubit routing.
3. Returns a Braket verbatim circuit ready for execution.

This replaces what would otherwise be a multi-step process of manually converting to Qiskit, looking up basis gates, compiling, and translating back. More details can be found in [this notebok](https://github.com/amazon-braket/amazon-braket-examples/blob/main/examples/qiskit/1_Compilation_with_the_Qiskit_Braket_provider.ipynb).

In [4]:
import os

from qiskit_braket_provider import to_braket

os.environ["BRAKET_DIAGRAM_WIDTH"] = "135"

verbatim_circuit = to_braket(braket_circuit, braket_device=device, optimization_level=1)

print("Compiled Braket verbatim circuit:")
print(verbatim_circuit)

Compiled Braket verbatim circuit:
T   : │        0        │         1         │         2         │  3  │         4         │         5         │  6  │         7         │         8         │  9  │        10         │        11         │  12  │      13       │ 14  │
                         ┌─────────────────┐ ┌─────────────────┐                                                                                                                                                                  ┌───┐ 
q13 : ───StartVerbatim───┤ PRx(1.57, 1.57) ├─┤ PRx(3.14, 0.00) ├───●─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────▒──────EndVerbatim───┤ M ├─
               ║         └─────────────────┘ └─────────────────┘   │                                                                                                                                                     ║        └───┘ 
               ║         ┌────────

/Users/maolinml/.julia/environments/v1.8/.CondaPkg/env/lib/python3.10/site-packages/qiskit_braket_provider/providers/adapter.py:1044: UserWarning: Device does not support global phase; global phase of 4.71238898038469 will not be included in Braket circuit
  warnings.warn(


## Run noisy emulation on the local emulator

The local emulator is created from the target device via `device.emulator()`. It automatically loads the latest calibration data (gate fidelities, readout errors, T1/T2 times) to build a device-aware noise model.

In [5]:
from collections import Counter

emulator = device.emulator()

SHOTS = 1000
result = emulator.run(verbatim_circuit, shots=SHOTS).result()
counts = Counter(result.measurement_counts)

print(f"Measurement counts ({SHOTS} shots):")
print(counts.most_common(10))

Measurement counts (1000 shots):
[('0000', 484), ('1111', 439), ('1110', 14), ('1101', 11), ('0010', 10), ('0111', 10), ('0001', 8), ('0100', 6), ('1000', 6), ('1011', 6)]


For an ideal GHZ state we expect only the all-zeros and all-ones bitstrings. The noisy emulator shows how device imperfections (gate errors, readout errors) spread probability to other bitstrings.

## Summary

In this notebook we demonstrated a complete workflow for:

1. **Building** a high-level quantum circuit with the Braket SDK.
2. **Compiling** it to a target QPU's native gate set using `to_braket()` with the `braket_device` parameter.
3. **Running** device-aware noisy emulation locally using the Braket device emulator.

This workflow is parameterized by `DEVICE_ARN`, so you can retarget it to any Braket QPU by changing a single variable. Because emulation runs entirely locally, it incurs zero cost — making it ideal for testing and validating circuits before submitting them to a QPU.

In [6]:
print("Tracker summary:")
print(t.quantum_tasks_statistics())
print(f"Estimated cost: ${t.simulator_tasks_cost():.2f}")

Tracker summary:
{}
Estimated cost: $0.00
